# Comparing small local LLMs: inference, sampling, quantization, and model size

In this notebook you run small open models on your own laptop with **Ollama**, send the same prompts to each one, and log what comes back: the outputs, the latency, and the token counts.

| Part | What you do | What you learn |
|---|---|---|
| 1 | Pull 4 small models (2 families × 2 sizes) | How model files are packaged (GGUF, parameter count, quantization tag) |
| 2 | Send the same prompts to every model | **Inference**: model load time, prompt processing (prefill), token generation (decode), tokens/sec |
| 3 | Change `temperature` and `top_p` | **Sampling**: how the next token is picked from the probability distribution |
| 4 | Run one model at q2 → q4 → q8 → fp16 | **Quantization**: trading size and speed against accuracy |
| 5 | Score every model on a small eval set | **Model size vs. quality**, plotted against speed |

Every call is appended to `runs/log_<timestamp>.jsonl`, so you can reload and reanalyze the results later.

---
## 0. Setup (one time, in a terminal)

```bash
brew install ollama          # or download the app from https://ollama.com/download
ollama serve                 # leave this running in its own terminal tab (the desktop app does this for you)
```

**About this hardware:** an Intel i7 Mac with no GPU acceleration, so everything runs on the CPU. On this machine, measured generation speed was about 12 tokens/s for 0.5B, 8 tokens/s for 1–1.5B, and 4 tokens/s for 3B. That's why the defaults below stay at 3B or under. You can add a 7–8B model (`qwen2.5:7b`, `llama3.1:8b`) at the end if you're willing to wait.

In [ ]:
import json, time, datetime, pathlib, re, difflib, random
from collections import Counter

import requests
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

OLLAMA = "http://localhost:11434"

LOG_DIR = pathlib.Path("runs")
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / "log_{:%Y%m%d_%H%M%S}.jsonl".format(datetime.datetime.now())

pd.set_option("display.max_colwidth", 120)

# Chart colors: fixed order, so a model keeps its color across every chart.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e4e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
})


def label_bars(ax, bars, fmt="{:.1f}", horizontal=False):
    # Write each bar's value at its end (Axes.bar_label needs matplotlib >= 3.4).
    for b in bars:
        if horizontal:
            v = b.get_width()
            ax.annotate(fmt.format(v), (v, b.get_y() + b.get_height() / 2), xytext=(4, 0),
                        textcoords="offset points", va="center", fontsize=9, color="#52514e")
        else:
            v = b.get_height()
            ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                        textcoords="offset points", ha="center", fontsize=8, color="#52514e")

### Helper functions

These helpers talk to Ollama's REST API directly with `requests`, so you can see exactly what goes over the wire. (The `ollama` Python package wraps the same endpoints.)

* `POST /api/pull` downloads a model.
* `POST /api/generate` runs one completion. Its response includes timing fields **in nanoseconds**:
  * `load_duration`: time to load the weights into RAM (large on the first call, about 0 after that while the model stays loaded)
  * `prompt_eval_count` / `prompt_eval_duration`: **prefill**, where the whole prompt is processed in one parallel pass
  * `eval_count` / `eval_duration`: **decode**, where one forward pass produces each output token. This is usually the bottleneck.

In [ ]:
def server_ok():
    try:
        v = requests.get(OLLAMA + "/api/version", timeout=3).json()["version"]
        print("Ollama is running, version", v)
        return True
    except requests.exceptions.ConnectionError:
        print("Can't reach Ollama at", OLLAMA, "- run `ollama serve` in a terminal (or open the Ollama app).")
        return False


def installed_models():
    """name -> metadata (size in bytes, details.parameter_size, details.quantization_level, ...)"""
    return {m["name"]: m for m in requests.get(OLLAMA + "/api/tags").json()["models"]}


def pull(model):
    if model in installed_models():
        print(model, "already downloaded")
        return
    with requests.post(OLLAMA + "/api/pull", json={"model": model}, stream=True) as r:
        r.raise_for_status()
        last = None
        for line in r.iter_lines():
            msg = json.loads(line)
            if "error" in msg:
                raise RuntimeError("{}: {}".format(model, msg["error"]))
            status = msg.get("status", "")
            if msg.get("total") and msg.get("completed") is not None:
                print("\r{}: {} {:5.1f}%".format(model, status, 100 * msg["completed"] / msg["total"]), end="")
            elif status != last:
                print("\n{}: {}".format(model, status), end="")
            last = status
    print()


DEFAULT_OPTIONS = {
    "temperature": 0.0,   # greedy decoding by default, so comparisons are repeatable
    "num_predict": 256,   # cap output length; CPU decode is slow
}


def generate(model, prompt, experiment="", system=None, **options):
    """Run one non-streaming completion, log it to LOG_FILE, and return a flat record."""
    opts = dict(DEFAULT_OPTIONS, **options)
    payload = {"model": model, "prompt": prompt, "stream": False, "options": opts, "keep_alive": "10m"}
    if system:
        payload["system"] = system

    t0 = time.perf_counter()
    r = requests.post(OLLAMA + "/api/generate", json=payload, timeout=900)
    wall = time.perf_counter() - t0
    r.raise_for_status()
    d = r.json()

    ns = 1e9
    rec = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "experiment": experiment,
        "model": model,
        "prompt": prompt,
        "options": opts,
        "response": d["response"],
        "done_reason": d.get("done_reason"),        # "stop" = finished naturally, "length" = hit num_predict
        "prompt_tokens": d.get("prompt_eval_count", 0),
        "output_tokens": d.get("eval_count", 0),
        "load_s": d.get("load_duration", 0) / ns,
        "prefill_s": d.get("prompt_eval_duration", 0) / ns,
        "decode_s": d.get("eval_duration", 0) / ns,
        "total_s": d.get("total_duration", 0) / ns,
        "wall_s": wall,
    }
    rec["decode_tok_per_s"] = rec["output_tokens"] / rec["decode_s"] if rec["decode_s"] else None
    rec["prefill_tok_per_s"] = rec["prompt_tokens"] / rec["prefill_s"] if rec["prefill_s"] else None
    # Approximate time-to-first-token: load + prefill (+ the first decode step)
    rec["ttft_s"] = rec["load_s"] + rec["prefill_s"]

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(rec) + "\n")
    return rec


def warm_up(model):
    """Load the weights once so that load time doesn't skew the first measured prompt."""
    return generate(model, "Hi", experiment="warmup", num_predict=1)


server_ok()

---
## Part 1: Pull the models

Two families at two sizes each. That lets you compare **size within a family** (0.5B vs 1.5B Qwen, 1B vs 3B Llama) and **family at similar size** (Llama 1B vs Qwen 1.5B).

Most default Ollama tags are **4-bit quantized (`Q4_K_M`)**. The exception is `llama3.2:1b`, which defaults to **8-bit (`Q8_0`)**; check the `quant` column below. Downloads total about 4.4 GB.

In [ ]:
MODELS = [
    "qwen2.5:0.5b",
    "llama3.2:1b",
    "qwen2.5:1.5b",
    "llama3.2:3b",
    # "qwen2.5:7b",   # optional: about 4.7 GB, slow on this CPU (~3 tok/s)
]

for m in MODELS:
    pull(m)

In [ ]:
def model_table(names):
    inst = installed_models()
    rows = []
    for n in names:
        if n not in inst:
            continue
        m = inst[n]
        rows.append({
            "model": n,
            "params": m["details"].get("parameter_size"),
            "quant": m["details"].get("quantization_level"),
            "format": m["details"].get("format"),
            "family": m["details"].get("family"),
            "size_GB": round(m["size"] / 1e9, 2),
        })
    return pd.DataFrame(rows)

model_table(MODELS)

**Check yourself:** size on disk ≈ parameters × bits-per-weight / 8. At Q4_K_M (about 4.8 bits/weight on average), a 3.2B model should come out near 3.2e9 × 4.8 / 8 ≈ 1.9 GB. Does the table match?

---
## Part 2: Same prompts, every model

The prompts mix task types on purpose (fact recall, reasoning, code, instruction following, creative writing), because small models fail unevenly across them.

In [ ]:
PROMPTS = {
    "explain":   "Explain what a hash table is to a beginner programmer in 3 sentences.",
    "reasoning": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Think step by step, then give the final answer.",
    "code":      "Write a Python function `is_palindrome(s)` that ignores case and non-alphanumeric characters. Code only.",
    "format":    "List three planets in our solar system as a JSON array of strings. Output only the JSON.",
    "creative":  "Write a haiku about debugging code at 2am.",
    "trick":     "What is the 5th word in this sentence: 'The quick brown fox jumps over the lazy dog'?",
}

results = []
for m in MODELS:
    warm_up(m)
    for name, p in PROMPTS.items():
        rec = generate(m, p, experiment="compare")
        rec["prompt_name"] = name
        results.append(rec)
        print("{:14s} {:10s} {:4d} tok  {:5.1f} tok/s  {:5.1f}s".format(
            m, name, rec["output_tokens"], rec["decode_tok_per_s"] or 0, rec["wall_s"]))

df = pd.DataFrame(results)

### Read the outputs side by side

In [ ]:
def show_side_by_side(df, prompt_name):
    sub = df[df.prompt_name == prompt_name]
    out = ["### `{}`\n> {}\n".format(prompt_name, sub.prompt.iloc[0])]
    for _, r in sub.iterrows():
        out.append("**{}** ({} tokens, {:.1f}s)\n\n```\n{}\n```\n".format(
            r.model, r.output_tokens, r.wall_s, r.response.strip()))
    display(Markdown("\n".join(out)))

for name in PROMPTS:
    show_side_by_side(df, name)

### Latency and throughput

In [ ]:
summary = (df.groupby("model", sort=False)
             .agg(prompt_tok=("prompt_tokens", "mean"),
                  output_tok=("output_tokens", "mean"),
                  prefill_tok_s=("prefill_tok_per_s", "mean"),
                  decode_tok_s=("decode_tok_per_s", "mean"),
                  ttft_s=("ttft_s", "mean"),
                  wall_s=("wall_s", "mean"))
             .round(2))
summary

In [ ]:
def model_bar(series, title, xlabel, ax=None):
    ax = ax or plt.gca()
    colors = [PALETTE[MODELS.index(m) % len(PALETTE)] if m in MODELS else PALETTE[0] for m in series.index]
    bars = ax.barh(series.index[::-1], series.values[::-1], color=colors[::-1], height=0.6)
    label_bars(ax, bars, horizontal=True)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
model_bar(summary.decode_tok_s, "Generation speed (decode)", "tokens / second", axes[0])
model_bar(summary.prefill_tok_s, "Prompt processing speed (prefill)", "tokens / second", axes[1])
plt.tight_layout()

**Things to notice:**
* **Prefill is much faster per token than decode.** Prefill processes every prompt token in one batched matrix multiply. Decode produces one token per forward pass and has to stream *all* the weights from RAM for each token. On a CPU, decode speed is roughly `memory bandwidth / model size in bytes`, which is why quantization speeds it up.
* Decode tok/s should fall roughly in proportion to parameter count.
* Check `done_reason` in `df`. A `"length"` means the model hit `num_predict` and was cut off.
* Same-size models from different families can differ in verbosity (`output_tok`), and that changes wall-clock latency even when tok/s is the same.
* Different families use different **tokenizers**, so the same prompt gives a different `prompt_tok` count for Llama and Qwen.

---
## Part 3: Sampling parameters

At every step the model outputs a probability distribution over its whole vocabulary (about 128k tokens for Llama 3.2, about 152k for Qwen 2.5). **Sampling** decides which token to pick from it:

* **temperature** rescales the logits before softmax: `p_i ∝ exp(logit_i / T)`. With `T → 0` the top token always wins (greedy decoding), `T = 1` leaves the distribution unchanged, and `T > 1` flattens it so unlikely tokens get picked more often.
* **top_p** (nucleus sampling) keeps the smallest set of tokens whose probabilities sum to `p` and samples only from that set. A low `top_p` cuts off the long tail.
* **top_k** keeps only the k most likely tokens. Ollama's default is 40; the cells below set it to 0 (off) so `top_p` acts alone.
* **seed** fixes the random number generator. The same seed with the same settings gives the same output.

All experiments in this part use one model so the only thing that changes is the sampling.

In [ ]:
SAMPLING_MODEL = "llama3.2:1b"
warm_up(SAMPLING_MODEL)

def distinct_n(texts, n=2):
    """Fraction of n-grams that are unique across all samples: higher means more diverse."""
    grams = []
    for t in texts:
        toks = t.lower().split()
        grams += [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)

### 3a. Is temperature 0 deterministic? Does a seed make sampling repeatable?

In [ ]:
p = "Write a one-sentence opening line for a mystery novel."
greedy = [generate(SAMPLING_MODEL, p, "determinism", temperature=0)["response"] for _ in range(3)]
seeded = [generate(SAMPLING_MODEL, p, "determinism", temperature=1.0, seed=7)["response"] for _ in range(3)]
unseeded = [generate(SAMPLING_MODEL, p, "determinism", temperature=1.0, seed=random.randint(0, 10**6))["response"] for _ in range(3)]

for label, outs in [("temperature=0", greedy), ("temperature=1, seed=7", seeded), ("temperature=1, random seeds", unseeded)]:
    print("== {}  ({} unique of {})".format(label, len(set(outs)), len(outs)))
    for o in outs:
        print("  -", o.strip()[:150])

### 3b. Temperature sweep

In [ ]:
TEMPS = [0.0, 0.5, 1.0, 1.5, 2.0]
N_SAMPLES = 5
p = "Write a one-sentence opening line for a mystery novel."

rows = []
for t in TEMPS:
    for i in range(N_SAMPLES):
        r = generate(SAMPLING_MODEL, p, "temp_sweep", temperature=t, top_p=1.0, top_k=0, seed=1000 + i, num_predict=60)
        rows.append({"temperature": t, "sample": i, "response": r["response"].strip()})
temp_df = pd.DataFrame(rows)

temp_stats = temp_df.groupby("temperature").response.agg(
    unique=lambda s: s.nunique(), distinct_2=lambda s: round(distinct_n(list(s)), 2))
display(temp_stats)

for t in TEMPS:
    display(Markdown("**T = {}**\n".format(t) + "\n".join("- " + r for r in temp_df[temp_df.temperature == t].response)))

At T = 1.5–2.0 with `top_k` off, you'll probably see text fall apart into unrelated words, odd Unicode, or another language. That happens because a flattened distribution gives real probability mass to thousands of junk tokens. This is why `top_p` / `top_k` are normally used together with temperature.

### 3c. top_p sweep (temperature fixed at 1.5)

In [ ]:
TOP_PS = [0.1, 0.5, 0.9, 1.0]
rows = []
for tp in TOP_PS:
    for i in range(N_SAMPLES):
        r = generate(SAMPLING_MODEL, p, "top_p_sweep", temperature=1.5, top_p=tp, top_k=0, seed=2000 + i, num_predict=60)
        rows.append({"top_p": tp, "sample": i, "response": r["response"].strip()})
topp_df = pd.DataFrame(rows)

display(topp_df.groupby("top_p").response.agg(
    unique=lambda s: s.nunique(), distinct_2=lambda s: round(distinct_n(list(s)), 2)))
for tp in TOP_PS:
    display(Markdown("**top_p = {}**\n".format(tp) + "\n".join("- " + r for r in topp_df[topp_df.top_p == tp].response)))

### 3d. Ask an LLM for a random number

LLMs are bad random number generators, and this experiment shows the model's preferred answer directly. Ask for "a random number between 1 and 10" 30 times at each temperature and plot the counts.

In [ ]:
p = "Pick a random number between 1 and 10. Reply with only the number."
counts = {}
for t in [0.7, 1.0, 1.5]:
    answers = []
    for i in range(30):
        r = generate(SAMPLING_MODEL, p, "random_number", temperature=t, seed=3000 + i, num_predict=4)
        m = re.search(r"\d+", r["response"])
        answers.append(int(m.group()) if m else None)
    counts[t] = Counter(a for a in answers if a is not None)

fig, axes = plt.subplots(1, len(counts), figsize=(11, 2.8), sharey=True)
for ax, (t, c) in zip(axes, counts.items()):
    xs = list(range(1, 11))
    ax.bar(xs, [c.get(x, 0) for x in xs], color=PALETTE[0], width=0.7)
    ax.set_title("temperature = {}".format(t), loc="left", fontsize=10)
    ax.set_xticks(xs)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("count (of 30)")
plt.tight_layout()

---
## Part 4: Quantization

Quantization stores each weight in fewer bits. Ollama models use the **GGUF** format from llama.cpp. Common levels:

| Tag | ≈ bits/weight | Notes |
|---|---|---|
| `fp16` | 16 | unquantized reference |
| `q8_0` | 8.5 | 8-bit blocks of 32 weights, each block with its own scale. Nearly lossless |
| `q4_K_M` | ~4.8 | "K-quant", mixed precision (sensitive layers get more bits). Ollama's default |
| `q4_0` | 4.5 | older, simpler 4-bit scheme |
| `q2_K` | ~3 | aggressive. Small models suffer badly at this level |

Smaller models are **more** sensitive to quantization than large ones, because they have less redundancy in their weights. Here you run the same 1B model at each level.

In [ ]:
QUANTS = [
    "llama3.2:1b-instruct-fp16",
    "llama3.2:1b-instruct-q8_0",     # same file as llama3.2:1b, so no extra download
    "llama3.2:1b-instruct-q4_K_M",
    "llama3.2:1b-instruct-q4_0",
    "llama3.2:1b-instruct-q2_K",
]
available_quants = []
for q in QUANTS:
    try:
        pull(q)
        available_quants.append(q)
    except Exception as e:
        print("skipping:", e)

model_table(available_quants)

Next, run a graded eval set: questions with one checkable right answer. You'll reuse it in Part 5. Each item pairs a prompt with a checker function.

In [ ]:
def number_is(x):
    def check(resp):
        m = re.search(r"-?\d+(?:\.\d+)?", resp.replace(",", ""))
        return bool(m) and float(m.group()) == float(x)
    return check

def contains(pattern):
    return lambda resp: re.search(pattern, resp, re.IGNORECASE) is not None

def is_json_person(resp):
    m = re.search(r"\{.*\}", resp, re.DOTALL)
    try:
        obj = json.loads(m.group()) if m else None
    except ValueError:
        return False
    return isinstance(obj, dict) and str(obj.get("name", "")).lower() == "ada" and str(obj.get("age")) == "36"

EVAL = [
    ("arith_mul",    "What is 17 * 24? Reply with only the number.",                                     number_is(408)),
    ("arith_pct",    "What is 15% of 80? Reply with only the number.",                                   number_is(12)),
    ("capital",      "What is the capital of Australia? Reply with one word.",                           contains(r"\bcanberra\b")),
    ("gold",         "What is the chemical symbol for gold? Reply with only the symbol.",                contains(r"\bAu\b")),
    ("author",       "Who wrote 'Pride and Prejudice'? Reply with only the author's name.",              contains(r"austen")),
    ("spanish",      "Translate 'thank you' into Spanish. Reply with only the translation.",             contains(r"gracias")),
    ("letters",      "How many times does the letter r appear in 'strawberry'? Reply with only the number.", number_is(3)),
    ("reverse",      "Reverse the letters of the word 'planet'. Reply with only the reversed word.",     contains(r"\btenalp\b")),
    ("sort",         "Sort alphabetically: pear, mango, apple, banana. Reply with the words comma-separated.", contains(r"apple,\s*banana,\s*mango,\s*pear")),
    ("feathers",     "Which is heavier: a kilogram of feathers or a kilogram of steel? Reply with one word: feathers, steel, or same.", contains(r"\bsame\b")),
    ("sisters",      "Sally has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have? Reply with only the number.", number_is(1)),
    ("time",         "A train leaves at 3:40 PM and the trip takes 2 hours 35 minutes. What time does it arrive? Reply in H:MM PM format.", contains(r"\b6:15\b")),
    ("json",         "Return a JSON object with keys 'name' and 'age' for a person named Ada who is 36. Output only JSON.", is_json_person),
    ("bat_ball",     "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How many cents does the ball cost? Reply with only the number.", number_is(5)),
    ("primes",       "How many prime numbers are there between 1 and 20? Reply with only the number.",  number_is(8)),
]

def run_eval(model, experiment):
    warm_up(model)
    rows = []
    for name, prompt, check in EVAL:
        r = generate(model, prompt, experiment, num_predict=64)
        rows.append({"model": model, "item": name, "correct": check(r["response"]),
                     "response": r["response"].strip()[:80],
                     "output_tokens": r["output_tokens"], "decode_tok_per_s": r["decode_tok_per_s"]})
    return rows

print(len(EVAL), "eval items")

In [ ]:
quant_rows = []
for q in available_quants:
    quant_rows += run_eval(q, "quant_eval")
    print("done", q)
quant_eval = pd.DataFrame(quant_rows)

# Also check how far each quantized model's greedy output drifts from fp16 on the open-ended prompts.
ref_model = available_quants[0]
ref = {name: generate(ref_model, p, "quant_drift")["response"] for name, p in PROMPTS.items()}
drift = []
for q in available_quants:
    for name, p in PROMPTS.items():
        r = generate(q, p, "quant_drift")
        drift.append({"model": q, "similarity_to_fp16": difflib.SequenceMatcher(None, ref[name], r["response"]).ratio(),
                      "decode_tok_per_s": r["decode_tok_per_s"]})
drift = pd.DataFrame(drift)

sizes = model_table(available_quants).set_index("model").size_GB
quant_summary = pd.DataFrame({
    "size_GB": sizes,
    "accuracy": quant_eval.groupby("model").correct.mean(),
    "decode_tok_s": drift.groupby("model").decode_tok_per_s.mean(),
    "similarity_to_fp16": drift.groupby("model").similarity_to_fp16.mean(),
}).loc[available_quants].round(2)
quant_summary

In [ ]:
short = [q.split("-")[-1] for q in quant_summary.index]
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, col, title in zip(axes,
                          ["size_GB", "decode_tok_s", "accuracy", "similarity_to_fp16"],
                          ["Size on disk (GB)", "Decode speed (tok/s)", "Eval accuracy", "Text similarity to fp16"]):
    bars = ax.bar(short, quant_summary[col], color=PALETTE[0], width=0.6)
    label_bars(ax, bars, fmt="{:.2f}")
    ax.set_title(title, loc="left", fontsize=10)
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()

In [ ]:
# Per-item correctness: which questions break first as bits go down?
quant_eval.pivot(index="item", columns="model", values="correct")[available_quants].replace({True: "✓", False: "·"})

**Things to notice:**
* Size scales almost linearly with bits per weight.
* Decode speed goes *up* as size goes down, because decode is memory-bandwidth bound.
* With a 15-item eval the accuracy numbers are noisy. A difference of one question is 7 percentage points, so look for big drops (often at q2_K), not small wiggles.
* "Similarity to fp16" usually stays high for q8_0. Lower-bit versions produce the same opening tokens and then *diverge*: one different token early on sends greedy decoding down a different path.

---
## Part 5: Model size vs. quality

Run the same eval on the models from Part 1 and plot accuracy against speed.

In [ ]:
size_rows = []
for m in MODELS:
    size_rows += run_eval(m, "size_eval")
    print("done", m)
size_eval = pd.DataFrame(size_rows)

info = model_table(MODELS).set_index("model")
size_summary = pd.DataFrame({
    "params": info.params,
    "size_GB": info.size_GB,
    "accuracy": size_eval.groupby("model").correct.mean(),
    "decode_tok_s": summary.decode_tok_s,
}).loc[MODELS].round(2)
size_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for i, (m, r) in enumerate(size_summary.iterrows()):
    ax.scatter(r.decode_tok_s, r.accuracy, s=40 + 60 * r.size_GB, color=PALETTE[i % len(PALETTE)],
               edgecolor="white", linewidth=2, zorder=3)
    ax.annotate("{}  ({})".format(m, r.params), (r.decode_tok_s, r.accuracy),
                xytext=(8, 4), textcoords="offset points", fontsize=9, color="#0b0b0b")
ax.set_xlabel("decode speed (tokens / s) →  faster")
ax.set_ylabel("eval accuracy →  better")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Speed vs. quality (marker area ∝ size on disk)", loc="left", fontsize=11)
plt.tight_layout()

In [ ]:
size_eval.pivot(index="item", columns="model", values="correct")[MODELS].replace({True: "✓", False: "·"})

**Things to notice:**
* Within a family, the larger model should beat the smaller one on reasoning items (`bat_ball`, `sisters`, `time`) while running proportionally slower.
* Some items fail for *every* small model whatever its size, especially `letters` and `reverse`. Models see **tokens**, not characters: `strawberry` might be a single token, so counting its letters means recalling the spelling from memory. A tokenizer explains this failure; model size doesn't.
* Fact recall (`capital`, `gold`, `author`) holds up well even at 0.5B. Multi-step reasoning goes first.

---
## Load past runs

Everything was logged to JSONL. To analyze across sessions, load all the log files:

In [ ]:
all_runs = pd.concat([pd.read_json(f, lines=True) for f in sorted(LOG_DIR.glob("*.jsonl"))], ignore_index=True)
print(len(all_runs), "logged calls")
all_runs[all_runs.experiment != "warmup"].groupby(["experiment", "model"]).size().unstack(fill_value=0)

---
## Exercises to try next

1. **System prompts:** pass `system="You are a terse assistant. Answer in as few words as possible."` to `generate` and rerun Part 2. How much do output tokens and latency drop, and does accuracy change?
2. **Context length and prefill:** paste a long document (2–4k tokens) into a prompt and watch `prefill_s` and `ttft_s` grow. Set `num_ctx` (default 2048 or 4096) higher, or the input gets silently truncated.
3. **A bigger model:** add `qwen2.5:7b` or `llama3.1:8b` to `MODELS`. Is the accuracy gain worth the drop to about 3 tok/s on this CPU?
4. **Quantize a larger model:** repeat Part 4 with `qwen2.5:3b-instruct-q2_K / q4_K_M / q8_0`. Is a 3B at q2 better or worse than a 1B at q8 of about the same file size?
5. **Streaming:** set `"stream": True` and measure the *real* time-to-first-token by timing the first chunk that arrives.
6. **llama.cpp directly:** `brew install llama.cpp` and run `llama-bench -m <path to a .gguf>` (Ollama keeps its blobs in `~/.ollama/models/blobs`) to get clean prefill and decode benchmarks.
7. **Logprobs:** check whether your Ollama version returns token log-probabilities (`"logprobs": true` in the request). If it does, compare the top-5 next-token distributions between fp16 and q2_K for the same prompt. That shows quantization noise directly.